In [0]:
# ========================================
# EXPLORATORY NOTEBOOK
# DATASET: gold_mart_customer_product_mix
# ========================================

# ========================================
# 0. CONFIGURATION
# ========================================

CATALOG = "ptfrozenfoods_dev"
SOURCE_SCHEMA = "gold"
TARGET_SCHEMA = "gold"

DOMAIN = "marts"
DATASET = "gold_mart_customer_product_mix"

STORAGE_ACCOUNT = "stptfrozenfoodsdevwe01"
GOLD_CONTAINER = "gold"

FACT_TABLE_NAME = "fact_sales"

FACT_TABLE = f"{CATALOG}.{SOURCE_SCHEMA}.{FACT_TABLE_NAME}"

CANDIDATE_TARGET_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.mart_customer_product_mix"

GOLD_BASE_PATH = f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

FACT_SOURCE_PATH = f"{GOLD_BASE_PATH}facts/fact_sales/"
CANDIDATE_TARGET_PATH = f"{GOLD_BASE_PATH}marts/mart_customer_product_mix/"

In [0]:
# ========================================
# 1. CONTEXT SETUP
# ========================================

print("=" * 80)
print("STARTING GOLD EXPLORATORY: gold_mart_customer_product_mix")
print("=" * 80)

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SOURCE_SCHEMA}")

print("[INFO] Context setup completed successfully.")

STARTING GOLD EXPLORATORY: gold_mart_customer_product_mix
[INFO] Context setup completed successfully.


In [0]:
# ========================================
# 2. CONFIGURATION SUMMARY
# ========================================

print("=" * 80)
print("GOLD EXPLORATORY NOTEBOOK CONFIGURATION")
print("=" * 80)
print(f"Catalog:                         {CATALOG}")
print(f"Source schema:                   {SOURCE_SCHEMA}")
print(f"Target schema:                   {TARGET_SCHEMA}")
print(f"Domain:                          {DOMAIN}")
print(f"Dataset:                         {DATASET}")
print(f"Fact table:                      {FACT_TABLE}")
print(f"Candidate target table:          {CANDIDATE_TARGET_TABLE}")
print(f"Fact source path:                {FACT_SOURCE_PATH}")
print(f"Candidate target path:           {CANDIDATE_TARGET_PATH}")
print("=" * 80)

GOLD EXPLORATORY NOTEBOOK CONFIGURATION
Catalog:                         ptfrozenfoods_dev
Source schema:                   gold
Target schema:                   gold
Domain:                          marts
Dataset:                         gold_mart_customer_product_mix
Fact table:                      ptfrozenfoods_dev.gold.fact_sales
Candidate target table:          ptfrozenfoods_dev.gold.mart_customer_product_mix
Fact source path:                abfss://gold@stptfrozenfoodsdevwe01.dfs.core.windows.net/facts/fact_sales/
Candidate target path:           abfss://gold@stptfrozenfoodsdevwe01.dfs.core.windows.net/marts/mart_customer_product_mix/


In [0]:
# ========================================
# 3. PRE-CHECKS
# ========================================

print("[INFO] Checking source table availability...")
spark.sql(f"DESCRIBE TABLE {FACT_TABLE}")

print("[INFO] Checking source path access...")
dbutils.fs.ls(FACT_SOURCE_PATH)

print("[INFO] Checking target container access...")
dbutils.fs.ls(f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/")

print("[INFO] Pre-checks completed successfully.")

[INFO] Checking source table availability...
[INFO] Checking source path access...
[INFO] Checking target container access...
[INFO] Pre-checks completed successfully.


In [0]:
# ========================================
# 4. READ SOURCE DATA
# ========================================

print("[INFO] Loading Gold source table...")

df_fact_sales = spark.table(FACT_TABLE)

print("[INFO] Source table loaded successfully.")

print("=" * 80)
print("SOURCE DATA ROW COUNTS")
print("=" * 80)
print(f"Fact sales:          {df_fact_sales.count():,}")
print("=" * 80)

[INFO] Loading Gold source table...
[INFO] Source table loaded successfully.
SOURCE DATA ROW COUNTS
Fact sales:          244,357


In [0]:
# ========================================
# DATASET PREVIEW — INITIAL EXPLORATION
# ========================================

print("=" * 80)
print("DATASET PREVIEW — INITIAL EXPLORATION")
print("=" * 80)

print("\n[INFO] Preview: FACT SALES")
print("-" * 80)
display(df_fact_sales.limit(5))

print("\n[INFO] Printing schema...")
df_fact_sales.printSchema()

print("\n" + "=" * 80)
print("[INFO] Dataset preview completed successfully.")
print("=" * 80)

DATASET PREVIEW — INITIAL EXPLORATION

[INFO] Preview: FACT SALES
--------------------------------------------------------------------------------


item_pedido_id,pedido_id,produto_id,cliente_id,canal_id,fornecedor_id,data_pedido,calendar_year,calendar_quarter,calendar_month,calendar_day,calendar_month_name,calendar_day_of_week,calendar_day_of_week_name,calendar_is_weekend,calendar_is_month_start,calendar_is_month_end,vendedor_id,cidade_entrega,estado_pedido,prazo_entrega_dias,tipo_cliente,cliente_cidade,distrito,segmento,potencial_valor,cluster_comercial,status_cliente,produto_nome,categoria,marca,status_produto,nome_canal,descricao_canal,quantity_sold,preco_lista_unitario,desconto_unitario,preco_venda_unitario,custo_unitario,gross_sales_amount,net_sales_amount,total_cost_amount,gross_margin_amount,line_count,flag_promocao,lote_fornecedor,item_load_date,item_ingestion_timestamp,item_source_file,order_load_date,order_ingestion_timestamp,order_source_file,average_order_value
IT000000027,PED2021011100012,P012,C0136,CH04,F006,2021-01-11,2021,1,1,11,Janeiro,1,Segunda-feira,0,0,0,UNKNOWN,Espinho,Entregue,1,Restaurante,Espinho,Aveiro,Pequeno,Alto,Cluster C,Ativo,Hambúrguer bovino 800g,Carne,Chef Express,Ativo,Marketplace,Plataformas externas de venda,2,5.49,0.0,5.49,3.14,10.98,10.98,6.28,4.7,1,0,L66962,2026-03-17,2026-04-03T22:27:43.144Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_order_items/load_date=2026-03-17/erp_order_items.csv,2026-03-17,2026-04-03T22:23:23.76Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_orders/load_date=2026-03-17/erp_orders.csv,29.42
IT000000026,PED2021011100012,P020,C0136,CH04,F002,2021-01-11,2021,1,1,11,Janeiro,1,Segunda-feira,0,0,0,UNKNOWN,Espinho,Entregue,1,Restaurante,Espinho,Aveiro,Pequeno,Alto,Cluster C,Ativo,Batata smile 1500g,Batatas,FrioMax,Ativo,Marketplace,Plataformas externas de venda,4,4.61,0.0,4.61,2.34,18.44,18.44,9.36,9.080000000000002,1,0,L39662,2026-03-17,2026-04-03T22:27:43.144Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_order_items/load_date=2026-03-17/erp_order_items.csv,2026-03-17,2026-04-03T22:23:23.76Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_orders/load_date=2026-03-17/erp_orders.csv,29.42
IT000000028,PED2021011100013,P004,C0136,CH02,F001,2021-01-11,2021,1,1,11,Janeiro,1,Segunda-feira,0,0,0,VEND001,Espinho,Entregue,1,Restaurante,Espinho,Aveiro,Pequeno,Alto,Cluster C,Ativo,Bacalhau desfiado 800g,Peixe,PT Frozen Foods,Ativo,Vendas Externas,Equipa comercial visitando clientes,2,8.72,0.0,8.72,4.9,17.44,17.44,9.8,7.640000000000001,1,0,L28888,2026-03-17,2026-04-03T22:27:43.144Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_order_items/load_date=2026-03-17/erp_order_items.csv,2026-03-17,2026-04-03T22:23:23.76Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_orders/load_date=2026-03-17/erp_orders.csv,53.79
IT000000030,PED2021011100013,P023,C0136,CH02,F005,2021-01-11,2021,1,1,11,Janeiro,1,Segunda-feira,0,0,0,VEND001,Espinho,Entregue,1,Restaurante,Espinho,Aveiro,Pequeno,Alto,Cluster C,Ativo,Pão de forma 800g,Padaria,Doce Norte,Ativo,Vendas Externas,Equipa comercial visitando clientes,4,2.97,0.15,2.82,1.36,11.88,11.28,5.44,5.839999999999999,1,1,L45385,2026-03-17,2026-04-03T22:27:43.144Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_order_items/load_date=2026-03-17/erp_order_items.csv,2026-03-17,2026-04-03T22:23:23.76Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_orders/load_date=2026-03-17/erp_orders.csv,53.79
IT000000029,PED2021011100013,P020,C0136,CH02,F002,2021-01-11,2021,1,1,11,Janeiro,1,Segunda-feira,0,0,0,VEND001,Espinho,Entregue,1,Restaurante,Espinho,Aveiro,Pequeno,Alto,Cluster C,Ativo,Batata smile 1500g,Batatas,FrioMax,Ativo,Vendas Externas,Equipa comercial visitando clientes,3,4.93,0.0,4.93,2.24,14.79,14.79,6.720000000000001,8.069999999999999,1,0,L68711,2026-03-17,2026-04-03T22:27:43.144Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_order_items/load_date=2026-03-17/erp_order_items.csv,2026-03-17,2026-04-03T22:23:23.76Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.c


[INFO] Printing schema...
root
 |-- item_pedido_id: string (nullable = true)
 |-- pedido_id: string (nullable = true)
 |-- produto_id: string (nullable = true)
 |-- cliente_id: string (nullable = true)
 |-- canal_id: string (nullable = true)
 |-- fornecedor_id: string (nullable = true)
 |-- data_pedido: date (nullable = true)
 |-- calendar_year: integer (nullable = true)
 |-- calendar_quarter: integer (nullable = true)
 |-- calendar_month: integer (nullable = true)
 |-- calendar_day: integer (nullable = true)
 |-- calendar_month_name: string (nullable = true)
 |-- calendar_day_of_week: integer (nullable = true)
 |-- calendar_day_of_week_name: string (nullable = true)
 |-- calendar_is_weekend: integer (nullable = true)
 |-- calendar_is_month_start: integer (nullable = true)
 |-- calendar_is_month_end: integer (nullable = true)
 |-- vendedor_id: string (nullable = true)
 |-- cidade_entrega: string (nullable = true)
 |-- estado_pedido: string (nullable = true)
 |-- prazo_entrega_dias: in

In [0]:
# ========================================
# 5. SOURCE VALIDATION
# ========================================

from pyspark.sql import functions as F

required_fact_columns = [
    "pedido_id",
    "data_pedido",
    "cliente_id",
    "produto_id",
    "canal_id",
    "fornecedor_id",
    "tipo_cliente",
    "cliente_cidade",
    "distrito",
    "segmento",
    "potencial_valor",
    "cluster_comercial",
    "status_cliente",
    "produto_nome",
    "categoria",
    "marca",
    "status_produto",
    "quantity_sold",
    "gross_sales_amount",
    "net_sales_amount",
    "total_cost_amount",
    "gross_margin_amount",
    "line_count",
    "average_order_value"
]

missing_fact_columns = [c for c in required_fact_columns if c not in df_fact_sales.columns]

print(f"[INFO] Missing fact columns: {missing_fact_columns}")

if missing_fact_columns:
    raise Exception("Missing required columns in fact_sales.")

print("[INFO] Source validation completed successfully.")

[INFO] Missing fact columns: []
[INFO] Source validation completed successfully.


In [0]:
# ========================================
# 6. INITIAL DATA QUALITY CHECKS
# ========================================

display(
    df_fact_sales.select(
        F.count("*").alias("total_rows"),
        F.countDistinct("data_pedido").alias("distinct_data_pedido"),
        F.countDistinct("cliente_id").alias("distinct_cliente_id"),
        F.countDistinct("produto_id").alias("distinct_produto_id"),
        F.sum(F.when(F.col("data_pedido").isNull(), 1).otherwise(0)).alias("null_data_pedido"),
        F.sum(F.when(F.col("cliente_id").isNull(), 1).otherwise(0)).alias("null_cliente_id"),
        F.sum(F.when(F.col("produto_id").isNull(), 1).otherwise(0)).alias("null_produto_id"),
        F.sum(F.when(F.col("net_sales_amount").isNull(), 1).otherwise(0)).alias("null_net_sales_amount"),
        F.sum(F.when(F.col("gross_margin_amount").isNull(), 1).otherwise(0)).alias("null_gross_margin_amount")
    )
)

total_rows,distinct_data_pedido,distinct_cliente_id,distinct_produto_id,null_data_pedido,null_cliente_id,null_produto_id,null_net_sales_amount,null_gross_margin_amount
244357,1875,145,36,0,0,0,0,0


In [0]:
# ========================================
# 7. BUSINESS PROFILE CHECKS
# ========================================

print("[INFO] Net sales by customer")
display(
    df_fact_sales.groupBy("cliente_id", "tipo_cliente", "segmento")
    .agg(
        F.round(F.sum("net_sales_amount"), 2).alias("total_net_sales"),
        F.round(F.sum("gross_margin_amount"), 2).alias("total_gross_margin"),
        F.sum("quantity_sold").alias("total_quantity_sold"),
        F.countDistinct("pedido_id").alias("total_orders")
    )
    .orderBy(F.desc("total_net_sales"))
)

print("[INFO] Net sales by product")
display(
    df_fact_sales.groupBy("produto_id", "produto_nome", "categoria", "marca")
    .agg(
        F.round(F.sum("net_sales_amount"), 2).alias("total_net_sales"),
        F.round(F.sum("gross_margin_amount"), 2).alias("total_gross_margin"),
        F.sum("quantity_sold").alias("total_quantity_sold"),
        F.countDistinct("pedido_id").alias("total_orders")
    )
    .orderBy(F.desc("total_net_sales"))
)

print("[INFO] Net sales by customer-product pairs")
display(
    df_fact_sales.groupBy("cliente_id", "produto_id")
    .agg(
        F.round(F.sum("net_sales_amount"), 2).alias("total_net_sales"),
        F.sum("quantity_sold").alias("total_quantity_sold"),
        F.countDistinct("pedido_id").alias("total_orders")
    )
    .orderBy(F.desc("total_net_sales"))
)

[INFO] Net sales by customer


cliente_id,tipo_cliente,segmento,total_net_sales,total_gross_margin,total_quantity_sold,total_orders
C0164,Restaurante,Grande,379267.11,143268.81,58090,4640
C0072,Supermercado,Grande,268120.63,105075.71,50331,3264
C0260,Hotel,Grande,245366.82,95384.94,40651,2971
C0314,Restaurante,Grande,237682.05,91556.13,37234,3076
C0103,Supermercado,Médio,224874.99,90476.0,43001,2981
C0031,Hotel,Grande,219373.69,88815.31,35372,2639
C0043,Supermercado,Grande,213785.07,83533.61,38796,2139
C0227,Hotel,Grande,177809.19,71891.69,28636,2815
C0139,Supermercado,Grande,177150.45,70521.67,34414,2241
C0057,Restaurante,Médio,173566.39,68361.06,26536,3876


[INFO] Net sales by product


produto_id,produto_nome,categoria,marca,total_net_sales,total_gross_margin,total_quantity_sold,total_orders
P004,Bacalhau desfiado 800g,Peixe,PT Frozen Foods,1038249.71,387513.65,121966,25720
P007,Cocktail de marisco 1kg,Marisco,FrioMax,541910.89,171906.95,55859,13691
P015,Lasanha bolonhesa 1kg,Pré-cozinhados,Atlântico Select,500059.9,173225.92,73042,13487
P006,Miolo de mexilhão 1kg,Marisco,FrioMax,401121.48,162017.69,53694,12048
P012,Hambúrguer bovino 800g,Carne,Chef Express,365556.18,151150.02,61862,16958
P020,Batata smile 1500g,Batatas,FrioMax,301340.79,139169.86,63635,16016
P014,Almôndegas de novilho 1kg,Carne,Mesa Pronta,210256.47,79797.32,34664,8243
P005,Miolo de camarão 60/80 800g,Marisco,Norte Mar,205950.54,63058.38,18773,5477
P017,Bacalhau com natas 1200g,Pré-cozinhados,PT Frozen Foods,201954.37,67798.69,22567,5801
P003,Potas em anéis 1kg,Peixe,Norte Mar,173512.32,69772.9,24988,6566


[INFO] Net sales by customer-product pairs


cliente_id,produto_id,total_net_sales,total_quantity_sold,total_orders
C0164,P004,103192.81,12482,2365
C0164,P007,63938.11,6804,1493
C0314,P004,57401.39,6878,1355
C0260,P004,56729.5,6812,1163
C0057,P004,48776.61,5851,1594
C0031,P004,47416.03,5573,1015
C0164,P006,42004.89,5786,1249
C0103,P004,41966.0,5032,837
C0043,P017,39133.6,4375,766
C0004,P004,38085.17,4378,926


In [0]:
# ========================================
# 8. BUILD CANDIDATE MART
# ========================================

df_mart_customer_product_mix_candidate = (
    df_fact_sales
    .groupBy(
        "data_pedido",
        "cliente_id",
        "produto_id",
        "tipo_cliente",
        "cliente_cidade",
        "distrito",
        "segmento",
        "potencial_valor",
        "cluster_comercial",
        "status_cliente",
        "produto_nome",
        "categoria",
        "marca",
        "status_produto"
    )
    .agg(
        F.round(F.sum("quantity_sold"), 2).alias("quantity_sold"),
        F.round(F.sum("gross_sales_amount"), 2).alias("gross_sales_amount"),
        F.round(F.sum("net_sales_amount"), 2).alias("net_sales_amount"),
        F.round(F.sum("total_cost_amount"), 2).alias("total_cost_amount"),
        F.round(F.sum("gross_margin_amount"), 2).alias("gross_margin_amount"),
        F.countDistinct("pedido_id").alias("order_count"),
        F.sum("line_count").alias("line_count"),
        F.round(F.avg("average_order_value"), 2).alias("average_order_value")
    )
)

print(f"[INFO] Candidate mart row count: {df_mart_customer_product_mix_candidate.count():,}")

display(df_mart_customer_product_mix_candidate.limit(10))

[INFO] Candidate mart row count: 180,649


data_pedido,cliente_id,produto_id,tipo_cliente,cliente_cidade,distrito,segmento,potencial_valor,cluster_comercial,status_cliente,produto_nome,categoria,marca,status_produto,quantity_sold,gross_sales_amount,net_sales_amount,total_cost_amount,gross_margin_amount,order_count,line_count,average_order_value
2024-05-10,C0314,P015,Restaurante,Guimarães,Braga,Grande,Baixo,Cluster B,Ativo,Lasanha bolonhesa 1kg,Pré-cozinhados,Atlântico Select,Ativo,7,51.38,48.3,31.78,16.52,1,1,54.29
2023-10-12,C0314,P012,Restaurante,Guimarães,Braga,Grande,Baixo,Cluster B,Ativo,Hambúrguer bovino 800g,Carne,Chef Express,Ativo,1,6.27,6.14,3.4,2.74,1,1,153.68
2023-11-17,C0004,P015,Restaurante,Vila Nova de Gaia,Porto,Grande,Médio,Cluster A,Ativo,Lasanha bolonhesa 1kg,Pré-cozinhados,Atlântico Select,Ativo,2,14.62,14.04,9.12,4.92,1,1,70.58
2025-09-01,C0001,P010,Take-away,Vila Nova de Gaia,Porto,Médio,Baixo,Cluster C,Ativo,Legumes para sopa 1kg,Hortícolas,Campo Verde,Ativo,1,2.86,2.86,1.41,1.45,1,1,71.89
2023-05-28,C0057,P008,Restaurante,Santo Tirso,Porto,Médio,Baixo,Cluster C,Ativo,Ervilhas 1kg,Hortícolas,Campo Verde,Ativo,3,6.78,6.36,3.18,3.18,1,1,20.84
2023-07-12,C0057,P012,Restaurante,Santo Tirso,Porto,Médio,Baixo,Cluster C,Ativo,Hambúrguer bovino 800g,Carne,Chef Express,Ativo,1,6.32,6.19,3.3,2.89,1,1,70.03
2024-06-21,C0176,P020,Hotel,Porto,Porto,Médio,Baixo,Cluster C,Dormante,Batata smile 1500g,Batatas,FrioMax,Ativo,4,21.17,19.85,10.43,9.42,1,2,36.67
2025-07-26,C0058,P004,Restaurante,Barcelos,Braga,Grande,Médio,Cluster D,Ativo,Bacalhau desfiado 800g,Peixe,PT Frozen Foods,Ativo,9,84.51,82.8,51.3,31.5,1,1,111.77
2021-02-03,C0136,P006,Restaurante,Espinho,Aveiro,Pequeno,Alto,Cluster C,Ativo,Miolo de mexilhão 1kg,Marisco,FrioMax,Inativo,2,14.77,14.77,8.25,6.52,2,2,40.82
2022-02-04,C0103,P008,Supermercado,Porto,Porto,Médio,Alto,Cluster C,Ativo,Ervilhas 1kg,Hortícolas,Campo Verde,Ativo,5,11.0,10.65,5.1,5.55,1,1,76.98


In [0]:
# ========================================
# 9. CANDIDATE MART GRAIN TEST
# ========================================

print("[INFO] Testing candidate grain: one record per data_pedido, cliente_id, produto_id")

display(
    df_mart_customer_product_mix_candidate.select(
        F.count("*").alias("total_rows"),
        F.countDistinct(
            F.concat_ws("||", F.col("data_pedido"), F.col("cliente_id"), F.col("produto_id"))
        ).alias("distinct_grain_keys")
    )
)

[INFO] Testing candidate grain: one record per data_pedido, cliente_id, produto_id


total_rows,distinct_grain_keys
180649,180649


In [0]:
# ========================================
# 10. CANDIDATE MART VALIDATION
# ========================================

display(
    df_mart_customer_product_mix_candidate.select(
        F.count("*").alias("total_rows"),
        F.countDistinct(
            F.concat_ws("||", F.col("data_pedido"), F.col("cliente_id"), F.col("produto_id"))
        ).alias("distinct_grain_keys"),
        F.sum(F.when(F.col("data_pedido").isNull(), 1).otherwise(0)).alias("null_data_pedido"),
        F.sum(F.when(F.col("cliente_id").isNull(), 1).otherwise(0)).alias("null_cliente_id"),
        F.sum(F.when(F.col("produto_id").isNull(), 1).otherwise(0)).alias("null_produto_id"),
        F.sum(F.when(F.col("net_sales_amount").isNull(), 1).otherwise(0)).alias("null_net_sales_amount"),
        F.sum(F.when(F.col("gross_margin_amount").isNull(), 1).otherwise(0)).alias("null_gross_margin_amount")
    )
)

print("[INFO] Candidate mart validation completed successfully.")

total_rows,distinct_grain_keys,null_data_pedido,null_cliente_id,null_produto_id,null_net_sales_amount,null_gross_margin_amount
180649,180649,0,0,0,0,0


[INFO] Candidate mart validation completed successfully.


In [0]:
# ========================================
# 11. METRIC RECONCILIATION CHECK
# ========================================

fact_totals = df_fact_sales.select(
    F.round(F.sum("quantity_sold"), 2).alias("fact_quantity_sold"),
    F.round(F.sum("gross_sales_amount"), 2).alias("fact_gross_sales_amount"),
    F.round(F.sum("net_sales_amount"), 2).alias("fact_net_sales_amount"),
    F.round(F.sum("total_cost_amount"), 2).alias("fact_total_cost_amount"),
    F.round(F.sum("gross_margin_amount"), 2).alias("fact_gross_margin_amount"),
    F.countDistinct("pedido_id").alias("fact_order_count"),
    F.sum("line_count").alias("fact_line_count")
)

candidate_totals = df_mart_customer_product_mix_candidate.select(
    F.round(F.sum("quantity_sold"), 2).alias("mart_quantity_sold"),
    F.round(F.sum("gross_sales_amount"), 2).alias("mart_gross_sales_amount"),
    F.round(F.sum("net_sales_amount"), 2).alias("mart_net_sales_amount"),
    F.round(F.sum("total_cost_amount"), 2).alias("mart_total_cost_amount"),
    F.round(F.sum("gross_margin_amount"), 2).alias("mart_gross_margin_amount"),
    F.sum("order_count").alias("mart_order_count"),
    F.sum("line_count").alias("mart_line_count")
)

display(fact_totals.crossJoin(candidate_totals))

fact_quantity_sold,fact_gross_sales_amount,fact_net_sales_amount,fact_total_cost_amount,fact_gross_margin_amount,fact_order_count,fact_line_count,mart_quantity_sold,mart_gross_sales_amount,mart_net_sales_amount,mart_total_cost_amount,mart_gross_margin_amount,mart_order_count,mart_line_count
914793,5780627.7,5487339.26,3328096.49,2159242.77,89526,244357,914793,5780627.7,5487339.26,3328096.49,2159242.77,216492,244357


%md
### Gold Layer Exploratory Conclusion — mart_customer_product_mix

This exploratory notebook validated the dataset required to build the `mart_customer_product_mix` mart in the Gold layer of the PT Frozen Foods data platform. The objective was to confirm data quality, validate dimensional assumptions, and ensure readiness for analytical consumption.

---

### Grain Validation

The mart grain was defined and validated as:

- **One record per date, customer, and product (`data_pedido`, `cliente_id`, `produto_id`).**

Validation results:

- Total rows: 180,649  
- Distinct grain keys: 180,649  
- Null values: 0  
- Duplicate values: 0  

This confirms that the dataset is unique, consistent, and aligned with the expected analytical structure.

---

### Business Value Assessment

This mart enables detailed analysis of customer purchasing behavior at the product level.

#### High-Value Analytical Capabilities

- **Customer Product Mix Analysis**
  - Identify which products each customer purchases over time.

- **Cross-Selling and Recommendation**
  - Supports identification of product affinities and bundling opportunities.

- **Customer Segmentation by Consumption**
  - Analyze behavior across `segmento`, `cluster_comercial`, and `tipo_cliente`.

- **Product Penetration Analysis**
  - Evaluate how widely each product is adopted across the customer base.

- **Revenue Concentration**
  - Identify dependency on specific customers or products.

---

### Data Quality Assessment

| Metric | Result |
|--------|--------|
| Total Records | 180,649 |
| Duplicate Records | 0 |
| Null Values | 0 |
| Schema Consistency | Validated |
| Grain Consistency | Confirmed |

All validations confirm the dataset’s reliability for analytical workloads.

---

### Metric Reconciliation

All additive metrics were successfully reconciled with the `fact_sales` table:

| Metric | Status |
|--------|--------|
| quantity_sold | Reconciled |
| gross_sales_amount | Reconciled |
| net_sales_amount | Reconciled |
| total_cost_amount | Reconciled |
| gross_margin_amount | Reconciled |
| line_count | Reconciled |

#### Order Count Behavior

| Source | Value |
|--------|-------|
| Distinct Orders in Fact Table | 89,526 |
| Aggregated Order Count in Mart | 216,492 |

The difference is expected.

> The `order_count` metric is non-additive at this grain. Orders containing multiple products are distributed across multiple records. Therefore, summing `order_count` does not represent the global number of distinct orders.

---

### Dimensional Model Alignment

The mart is fully aligned with the dimensional model and integrates:

- `fact_sales`
- Customer attributes
- Product attributes

It supports a star schema optimized for analytical queries and downstream consumption.

---

### Medallion Architecture Alignment

| Layer | Status |
|-------|--------|
| Bronze | Completed |
| Silver | Completed |
| Gold | Validated |

---

### Governance and Technology Alignment

| Component | Technology |
|-----------|------------|
| Storage | ADLS Gen2 |
| Format | Delta Lake |
| Processing | Azure Databricks |
| Governance | Unity Catalog |
| Orchestration | Azure Data Factory |

---

### Key Decisions

| Decision | Outcome |
|----------|---------|
| Mart grain | Defined as `data_pedido`, `cliente_id`, `produto_id` |
| Data source | Directly from `fact_sales` (optimized approach) |
| Joins | Eliminated (attributes already available) |
| Data quality | Fully validated |
| Metric reconciliation | Confirmed |
| Order count handling | Accepted as non-additive |
| Gold layer readiness | Confirmed |
| Performance strategy | Minimized transformations and scans |

---

### Conclusion

The dataset has been validated and approved for Gold layer processing. The analysis ensures:

- data quality and integrity;
- correct grain definition for customer-product analysis;
- adherence to dimensional modeling best practices;
- efficient processing with minimal transformations;
- readiness for BI, advanced analytics, and machine learning use cases.

The platform is now ready for the implementation of the `mart_customer_product_mix` Gold processing notebook.